<a href="https://colab.research.google.com/github/robsongfk/SalesInsightPY/blob/main/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# =====================================================================
# RF01 - Criar ou Carregar o Dataset de Vendas (Código exato do PDF)
# =====================================================================
def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headset"]
    categorias = {
        "Notebook": "Computadores", "Smartphone": "Celulares",
        "Tablet": "Celulares", "Monitor": "Computadores",
        "Teclado": "Perifericos", "Mouse": "Perifericos", "Headset": "Perifericos"
    }
    precos = {
        "Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
        "Monitor": 1200, "Teclado": 250, "Mouse": 120, "Headset": 350
    }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)

    dados = []
    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza
        if random.random() < 0.05: quantidade = None # valor nulo
        if random.random() < 0.04: preco = None # valor nulo
        if random.random() < 0.06: produto = " " + produto + " " # espacos extras
        if random.random() < 0.03: data_txt = "DATA INVALIDA" # data invalida
        if random.random() < 0.10: # ruido no nome
            cliente = random.choice([
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                " " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

# Gerar e salvar o CSV bruto
if not os.path.exists("vendas.csv"):
    df_bruto = gerar_dataset_vendas()
    df_bruto.to_csv("vendas.csv", index=False)
    print(f"Dataset gerado com {len(df_bruto)} registros.\n")


# =====================================================================
# RF02 - Inspecionar e Descrever os Dados
# =====================================================================
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

# Testando as etapas 1 e 2
df_carregado = pd.read_csv("vendas.csv")
inspecionar_dados(df_carregado)

Dataset gerado com 200 registros.

=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        10
preco_unitario     4
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14
...,...,...,...,...,...,...,...,...
195,196,2025-06-04,CLIENTE-038,Monitor,Computadores,Centro-Oeste,5.0,NaN
196,197,2025-09-15,Cliente_046,Tablet,Celulares,Sul,3.0,1748.92
197,198,2025-08-21,Cliente_048,Smartphone,Celulares,Sul,3.0,2185.60
198,199,2025-11-02,Cliente_006,Headset,Perifericos,Norte,4.0,368.94


In [2]:
import re
import numpy as np

# =====================================================================
# RF03 - Limpar e Tratar os Dados (datetime e regex)
# =====================================================================
def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna: (df_limpo, relatorio)
    """
    df_limpo = df.copy()
    qtd_inicial = len(df_limpo)

    # 1. Remover espaços extras nas colunas de texto
    colunas_texto = ['cliente', 'produto', 'categoria', 'regiao']
    for col in colunas_texto:
        df_limpo[col] = df_limpo[col].astype(str).str.strip()

    # 2. Converter data_venda e descartar datas inválidas (NaT)
    df_limpo['data_venda'] = pd.to_datetime(df_limpo['data_venda'], errors='coerce')
    qtd_com_datas = len(df_limpo.dropna(subset=['data_venda']))
    removidos_data = len(df_limpo) - qtd_com_datas
    df_limpo.dropna(subset=['data_venda'], inplace=True)

    # 3. Descartar nulos em quantidade e preco_unitario
    qtd_antes_nulos = len(df_limpo)
    df_limpo.dropna(subset=['quantidade', 'preco_unitario'], inplace=True)
    removidos_nulos = qtd_antes_nulos - len(df_limpo)

    # 4. Ajustar os tipos numéricos
    df_limpo['quantidade'] = df_limpo['quantidade'].astype(int)
    df_limpo['preco_unitario'] = df_limpo['preco_unitario'].astype(float)

    # 5. Padronizar o nome do cliente usando expressões regulares (regex)
    def tratar_cliente(c):
        # Mantém apenas letras, números e underscore
        c_limpo = re.sub(r"[^A-Za-z0-9_]", "", str(c).strip())
        # Extrai os números e força o padrão Cliente_NNN
        numeros = re.findall(r"\d+", c_limpo)
        if numeros:
            return f"Cliente_{int(numeros[0]):03d}"
        return c_limpo

    df_limpo['cliente'] = df_limpo['cliente'].apply(tratar_cliente)

    # 6. Montar o relatório de limpeza
    qtd_final = len(df_limpo)
    relatorio = {
        "Registros Iniciais": qtd_inicial,
        "Removidos (Data Inválida)": removidos_data,
        "Removidos (Valores Nulos)": removidos_nulos,
        "Registros Finais": qtd_final
    }

    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"{chave}: {valor}")

    return df_limpo, relatorio

# =====================================================================
# RF04 - Criar Colunas Derivadas com Transformações Condicionais
# =====================================================================
def criar_colunas_derivadas(df):
    """
    Cria colunas derivadas no DataFrame limpo.
    """
    # Operação vetorizada para receita total
    df['receita_total'] = df['quantidade'] * df['preco_unitario']

    # Colunas temporais
    df['mes'] = df['data_venda'].dt.month

    # Dicionário de mapeamento para o nome do mês em português
    meses_pt = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    df['mes_nome'] = df['mes'].map(meses_pt)
    df['trimestre'] = "Q" + df['data_venda'].dt.quarter.astype(str)
    df['ano'] = df['data_venda'].dt.year

    # Transformação condicional com np.select
    condicoes = [
        df['receita_total'] < 500,
        (df['receita_total'] >= 500) & (df['receita_total'] < 5000),
        df['receita_total'] >= 5000
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
    df['faixa_receita_item'] = np.select(condicoes, faixas, default="Nao Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===")
    print("Colunas atuais:", list(df.columns))

    return df

# Testando as funções com o DataFrame que carregamos na etapa anterior
df_limpo, relatorio_limpeza = limpar_dados(df_carregado)
df_transformado = criar_colunas_derivadas(df_limpo)

# Visualizando os primeiros registros para conferir o resultado
display(df_transformado.head())


=== RELATÓRIO DE LIMPEZA ===
Registros Iniciais: 200
Removidos (Data Inválida): 4
Removidos (Valores Nulos): 13
Registros Finais: 183

=== COLUNAS DERIVADAS CRIADAS ===
Colunas atuais: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario', 'receita_total', 'mes', 'mes_nome', 'trimestre', 'ano', 'faixa_receita_item']


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario,receita_total,mes,mes_nome,trimestre,ano,faixa_receita_item
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90,205.80,5,Maio,Q2,2025,Baixo Valor
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1,1939.76,1939.76,3,Março,Q1,2025,Medio Valor
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6,3864.87,23189.22,11,Novembro,Q4,2025,Alto Valor
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10,2008.14,20081.40,7,Julho,Q3,2025,Alto Valor
5,6,2025-08-21,Cliente_041,Headset,Perifericos,Sudeste,2,337.41,674.82,8,Agosto,Q3,2025,Medio Valor


In [3]:
!rm vendas.csv

In [4]:
# =====================================================================
# RF05 - Calcular Métricas Agregadas com groupby
# =====================================================================
def calcular_metricas(df):
    """
    Calcula as metricas agregadas do dataset.
    Retorna um dicionario com os DataFrames resultantes.
    """
    metricas = {}

    # 1. Por mês: Receita total, quantidade e número de vendas
    metricas['por_mes'] = df.groupby('mes').agg(
        receita_total=('receita_total', 'sum'),
        quantidade=('quantidade', 'sum'),
        n_vendas=('id_venda', 'count')
    ).reset_index()

    # 2. Top 5 Produtos por receita
    metricas['top_produtos'] = df.groupby('produto')['receita_total'].sum().sort_values(ascending=False).head(5).reset_index()

    # 3. Por Categoria
    metricas['por_categoria'] = df.groupby('categoria')['receita_total'].sum().reset_index()

    # 4. Por Região: Receita e Ticket Médio
    metricas['por_regiao'] = df.groupby('regiao').agg(
        receita_total=('receita_total', 'sum'),
        ticket_medio=('receita_total', 'mean')
    ).reset_index()

    print("\n=== MÉTRICAS AGREGADAS ===")
    print("--- POR MÊS ---")
    display(metricas['por_mes'].head())
    print("\n--- TOP PRODUTOS ---")
    display(metricas['top_produtos'])

    return metricas

# =====================================================================
# RF06 - Segmentar Clientes por Nível de Gasto
# =====================================================================
def segmentar_clientes(df):
    """
    Agrupa por cliente, soma a receita e classifica em Bronze/Prata/Ouro.
    """
    # Agrupa e calcula total gasto
    clientes = df.groupby('cliente')['receita_total'].sum().reset_index()
    clientes.rename(columns={'receita_total': 'total_gasto'}, inplace=True)

    # Função Lambda para classificar
    classifica_segmento = lambda gasto: 'Bronze' if gasto < 5000 else ('Prata' if gasto <= 15000 else 'Ouro')

    # Aplica a função lambda na coluna
    clientes['segmento'] = clientes['total_gasto'].apply(classifica_segmento)

    print("\n=== SEGMENTAÇÃO DE CLIENTES ===")
    print("--- Distribuição ---")
    print(clientes['segmento'].value_counts())
    print("\n--- Top 10 Clientes ---")
    display(clientes.sort_values(by='total_gasto', ascending=False).head(10))

    return clientes

# =====================================================================
# RF07 - Operações Numéricas com NumPy
# =====================================================================
def calcular_estatisticas_numpy(df):
    """
    Aplica operacoes NumPy sobre a coluna receita_total.
    """
    # Conversão para array
    receitas = df['receita_total'].to_numpy()

    # Funções de agregação NumPy
    media = np.mean(receitas)
    desvio = np.std(receitas) # ddof=0 por padrão
    maximo = np.max(receitas)
    minimo = np.min(receitas)

    # Broadcasting: Escalonamento para o intervalo 0-1
    receitas_escalonadas = (receitas - minimo) / (maximo - minimo)

    # Filtragem booleana sem laço for
    vendas_acima_media = receitas[receitas > media]

    estatisticas = {
        "media": media,
        "desvio_padrao": desvio,
        "maximo": maximo,
        "minimo": minimo,
        "qtd_vendas_acima_media": len(vendas_acima_media)
    }

    print("\n=== ESTATÍSTICAS NUMPY ===")
    for k, v in estatisticas.items():
        print(f"{k}: {v:.2f}")

    return estatisticas

# =====================================================================
# Testando as funções
# =====================================================================
metricas = calcular_metricas(df_transformado)
clientes_segmentados = segmentar_clientes(df_transformado)
estatisticas_gerais = calcular_estatisticas_numpy(df_transformado)


=== MÉTRICAS AGREGADAS ===
--- POR MÊS ---


,mes,receita_total,quantidade,n_vendas
0,1,120866.25,92,15
1,2,73895.74,70,12
2,3,123869.23,98,18
3,4,80636.64,48,7
4,5,132080.62,104,19



--- TOP PRODUTOS ---


,produto,receita_total
0,Notebook,374174.85
1,Tablet,335335.83
2,Smartphone,303255.94
3,Monitor,169554.67
4,Headset,48368.37



=== SEGMENTAÇÃO DE CLIENTES ===
--- Distribuição ---
segmento
Ouro      34
Prata     10
Bronze     6
Name: count, dtype: int64

--- Top 10 Clientes ---


,cliente,total_gasto,segmento
5,Cliente_006,65905.09,Ouro
35,Cliente_036,63433.04,Ouro
38,Cliente_039,59406.90,Ouro
19,Cliente_020,58459.54,Ouro
10,Cliente_011,56391.77,Ouro
33,Cliente_034,52035.60,Ouro
8,Cliente_009,50827.96,Ouro
23,Cliente_024,49146.94,Ouro
36,Cliente_037,46897.17,Ouro
18,Cliente_019,40731.05,Ouro



=== ESTATÍSTICAS NUMPY ===
media: 7051.07
desvio_padrao: 7809.92
maximo: 38833.20
minimo: 104.19
qtd_vendas_acima_media: 70.00


In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =====================================================================
# RF08 - Criar Visualizações com Matplotlib e Seaborn
# =====================================================================
def criar_visualizacoes(df, metricas):
    """Gera e exporta as visualizações em PNG."""

    # Cria a pasta para salvar os gráficos
    os.makedirs("outputs/graficos", exist_ok=True)

    # Configuração visual global (tema e tamanho padrão)
    sns.set_theme(style="whitegrid")
    plt.rcParams["figure.figsize"] = (10, 6)

    # 1. Gráfico de Linha: Receita por Mês
    fig, ax = plt.subplots()
    sns.lineplot(data=metricas['por_mes'], x='mes', y='receita_total', marker='o', linewidth=2, color='b', ax=ax)
    ax.set_title("Receita Total por Mês", fontsize=14)
    ax.set_xlabel("Mês")
    ax.set_ylabel("Receita Total (R$)")
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.savefig("outputs/graficos/receita_por_mes.png", dpi=150)
    plt.close() # Fecha a figura para não encavalar com a próxima

    # 2. Gráfico de Barras: Top 5 Produtos
    fig, ax = plt.subplots()
    sns.barplot(data=metricas['top_produtos'], y='produto', x='receita_total', hue='produto', legend=False, palette='viridis', ax=ax)
    ax.set_title("Top 5 Produtos por Receita", fontsize=14)
    ax.set_xlabel("Receita Total (R$)")
    ax.set_ylabel("Produto")
    plt.tight_layout()
    plt.savefig("outputs/graficos/top_produtos.png", dpi=150)
    plt.close()

    # 3. Gráfico de Dispersão: Quantidade vs Receita
    fig, ax = plt.subplots()
    sns.scatterplot(data=df, x='quantidade', y='receita_total', hue='categoria', palette='deep', alpha=0.8, ax=ax)
    ax.set_title("Relação: Quantidade Vendida vs Receita", fontsize=14)
    ax.set_xlabel("Quantidade")
    ax.set_ylabel("Receita Total (R$)")
    # Move a legenda para fora do gráfico para não tampar os dados
    plt.legend(title="Categoria", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig("outputs/graficos/quantidade_vs_receita.png", dpi=150)
    plt.close()

    # 4. Painel Resumo (Subplots 2x2 combinando os gráficos)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("SalesInsight PY - Painel Resumo", fontsize=18, fontweight='bold')

    # 4.1 Linha
    sns.lineplot(ax=axes[0, 0], data=metricas['por_mes'], x='mes', y='receita_total', marker='o', color='b')
    axes[0, 0].set_title("Receita por Mês")

    # 4.2 Barras (Top Produtos)
    sns.barplot(ax=axes[0, 1], data=metricas['top_produtos'], y='produto', x='receita_total', hue='produto', legend=False, palette='viridis')
    axes[0, 1].set_title("Top Produtos")

    # 4.3 Dispersão
    sns.scatterplot(ax=axes[1, 0], data=df, x='quantidade', y='receita_total', hue='categoria', palette='deep')
    axes[1, 0].set_title("Quantidade vs Receita")
    axes[1, 0].legend(title="Categoria", fontsize=9)

    # 4.4 Barras (Receita por Região)
    sns.barplot(ax=axes[1, 1], data=metricas['por_regiao'], x='regiao', y='receita_total', hue='regiao', legend=False, palette='magma')
    axes[1, 1].set_title("Receita por Região")

    # Ajusta o layout e salva o painel final
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("outputs/graficos/painel_resumo.png", dpi=150)
    plt.close()

    print("\n=== VISUALIZAÇÕES GERADAS COM SUCESSO ===")
    print("Abra a aba 'Arquivos' no menu esquerdo do Colab para ver a pasta 'outputs/graficos/'.")

# 5. Executa a função passando os dados que já estão na memória
criar_visualizacoes(df_transformado, metricas)


=== VISUALIZAÇÕES GERADAS COM SUCESSO ===
Abra a aba 'Arquivos' no menu esquerdo do Colab para ver a pasta 'outputs/graficos/'.


In [8]:
# =====================================================================
# RF09 (Parte A) - Função de Ordem Superior
# =====================================================================
def processar_coluna(df, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma função de transformação a uma coluna do DataFrame.
    Demonstra o uso de funções passadas como argumento.
    """
    nome_saida = nome_saida or f"{coluna}_transformado"
    df[nome_saida] = df[coluna].apply(funcao_transformacao)
    return df

# =====================================================================
# RF09 (Parte B) - Classe para organizar o fluxo
# =====================================================================
class AnalisadorDeVendas:
    """Encapsula o fluxo de análise dos dados de vendas."""

    def __init__(self, caminho_arquivo):
        # Atributos para guardar o "estado" do nosso projeto
        self.caminho_arquivo = caminho_arquivo
        self.df_bruto = None
        self.df_limpo = None
        self.metricas = {}
        self.clientes = None
        self.estatisticas = {}
        self.relatorio_limpeza = {}

    def carregar(self):
        """Lê o CSV e guarda o DataFrame bruto."""
        self.df_bruto = pd.read_csv(self.caminho_arquivo)
        print(f"[Analisador] {len(self.df_bruto)} registros carregados de '{self.caminho_arquivo}'.")

    def limpar(self):
        """Limpa os dados reaproveitando a função criada na Etapa 3."""
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto)
        print("[Analisador] Dados limpos com sucesso.")

    def transformar(self):
        """Cria colunas derivadas e testa a função de ordem superior."""
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)

        # Testando a função de ordem superior com um lambda para criar um perfil de volume
        self.df_limpo = processar_coluna(
            self.df_limpo,
            "quantidade",
            lambda q: "Alto Volume" if q >= 5 else "Baixo Volume",
            nome_saida="perfil_volume"
        )
        print("[Analisador] Transformações concluídas.")

    def analisar(self):
        """Calcula métricas, segmentação e operações NumPy (Etapas 5, 6 e 7)."""
        self.metricas = calcular_metricas(self.df_limpo)
        self.clientes = segmentar_clientes(self.df_limpo)
        self.estatisticas = calcular_estatisticas_numpy(self.df_limpo)
        print("[Analisador] Análises e métricas calculadas.")

    def visualizar(self):
        """Gera e exporta as figuras (Etapa 8)."""
        criar_visualizacoes(self.df_limpo, self.metricas)
        print("[Analisador] Visualizações geradas.")

    def resumo(self):
        """Imprime um resumo executivo do processamento."""
        print("\n" + "="*50)
        print("RESUMO EXECUTIVO DO PIPELINE")
        print("="*50)
        print(f"Total de registros úteis: {len(self.df_limpo)}")
        if not self.metricas['top_produtos'].empty:
            print(f"Produto Campeão de Receita: {self.metricas['top_produtos'].iloc[0]['produto']}")
        print("="*50)

# =====================================================================
# Testando a Classe AnalisadorDeVendas
# =====================================================================
print("--- TESTANDO A CLASSE ---")
analisador = AnalisadorDeVendas("vendas.csv")

# Chamando os métodos na ordem lógica (o Pipeline)
analisador.carregar()
analisador.limpar()
analisador.transformar()
analisador.analisar()
analisador.visualizar()
analisador.resumo()

--- TESTANDO A CLASSE ---
[Analisador] 200 registros carregados de 'vendas.csv'.

=== RELATÓRIO DE LIMPEZA ===
Registros Iniciais: 200
Removidos (Data Inválida): 4
Removidos (Valores Nulos): 13
Registros Finais: 183
[Analisador] Dados limpos com sucesso.

=== COLUNAS DERIVADAS CRIADAS ===
Colunas atuais: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario', 'receita_total', 'mes', 'mes_nome', 'trimestre', 'ano', 'faixa_receita_item']
[Analisador] Transformações concluídas.

=== MÉTRICAS AGREGADAS ===
--- POR MÊS ---


,mes,receita_total,quantidade,n_vendas
0,1,120866.25,92,15
1,2,73895.74,70,12
2,3,123869.23,98,18
3,4,80636.64,48,7
4,5,132080.62,104,19



--- TOP PRODUTOS ---


,produto,receita_total
0,Notebook,374174.85
1,Tablet,335335.83
2,Smartphone,303255.94
3,Monitor,169554.67
4,Headset,48368.37



=== SEGMENTAÇÃO DE CLIENTES ===
--- Distribuição ---
segmento
Ouro      34
Prata     10
Bronze     6
Name: count, dtype: int64

--- Top 10 Clientes ---


,cliente,total_gasto,segmento
5,Cliente_006,65905.09,Ouro
35,Cliente_036,63433.04,Ouro
38,Cliente_039,59406.90,Ouro
19,Cliente_020,58459.54,Ouro
10,Cliente_011,56391.77,Ouro
33,Cliente_034,52035.60,Ouro
8,Cliente_009,50827.96,Ouro
23,Cliente_024,49146.94,Ouro
36,Cliente_037,46897.17,Ouro
18,Cliente_019,40731.05,Ouro



=== ESTATÍSTICAS NUMPY ===
media: 7051.07
desvio_padrao: 7809.92
maximo: 38833.20
minimo: 104.19
qtd_vendas_acima_media: 70.00
[Analisador] Análises e métricas calculadas.

=== VISUALIZAÇÕES GERADAS COM SUCESSO ===
Abra a aba 'Arquivos' no menu esquerdo do Colab para ver a pasta 'outputs/graficos/'.
[Analisador] Visualizações geradas.

RESUMO EXECUTIVO DO PIPELINE
Total de registros úteis: 183
Produto Campeão de Receita: Notebook


In [9]:
import json
import os
import pandas as pd

# =====================================================================
# RF10 - Adicionando a Exportação na Classe AnalisadorDeVendas
# =====================================================================
class AnalisadorDeVendas:
    """Encapsula o fluxo de análise dos dados de vendas."""

    def __init__(self, caminho_arquivo):
        self.caminho_arquivo = caminho_arquivo
        self.df_bruto = None
        self.df_limpo = None
        self.metricas = {}
        self.clientes = None
        self.estatisticas = {}
        self.relatorio_limpeza = {}

    def carregar(self):
        self.df_bruto = pd.read_csv(self.caminho_arquivo)

    def limpar(self):
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto)

    def transformar(self):
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)
        self.df_limpo = processar_coluna(
            self.df_limpo, "quantidade", lambda q: "Alto Volume" if q >= 5 else "Baixo Volume", "perfil_volume"
        )

    def analisar(self):
        self.metricas = calcular_metricas(self.df_limpo)
        self.clientes = segmentar_clientes(self.df_limpo)
        self.estatisticas = calcular_estatisticas_numpy(self.df_limpo)

    def visualizar(self):
        criar_visualizacoes(self.df_limpo, self.metricas)

    # ---> NOVO MÉTODO ADICIONADO AQUI <---
    def exportar(self):
        """Exporta os resultados em CSV e JSON, e os lê novamente para conferência."""
        # 1. Cria a pasta outputs, se não existir
        os.makedirs("outputs", exist_ok=True)

        # 2. Exportando DataFrames para CSV
        self.metricas["por_mes"].to_csv("outputs/metricas_por_mes.csv", index=False, encoding="utf-8-sig")
        self.clientes.to_csv("outputs/segmentacao_clientes.csv", index=False, encoding="utf-8-sig")

        # 3. Preparando e gravando o JSON
        # O NumPy usa tipos de dados próprios (ex: int64, float64) que o JSON não entende nativamente.
        # Por isso, convertemos os valores para float nativo do Python antes de salvar.
        serializavel = {k: round(float(v), 2) for k, v in self.estatisticas.items()}
        caminho_json = "outputs/estatisticas_gerais.json"

        with open(caminho_json, "w", encoding="utf-8") as f:
            json.dump(serializavel, f, indent=4, ensure_ascii=False)

        # 4. Lendo o JSON de volta para confirmar a gravação (Exigência do RF10)
        with open(caminho_json, "r", encoding="utf-8") as f:
            conferencia = json.load(f)

        print("\n=== EXPORTAÇÃO DE ARQUIVOS ===")
        print(f"JSON gravado e lido com sucesso:\n{json.dumps(conferencia, indent=2)}")
        print("Arquivos CSV salvos na pasta 'outputs/'.")

# =====================================================================
# Testando a Exportação
# =====================================================================
print("Rodando o pipeline até a exportação...")
analisador = AnalisadorDeVendas("vendas.csv")
analisador.carregar()
analisador.limpar()
analisador.transformar()
analisador.analisar()
analisador.exportar() # Executa a nossa nova função

Rodando o pipeline até a exportação...

=== RELATÓRIO DE LIMPEZA ===
Registros Iniciais: 200
Removidos (Data Inválida): 4
Removidos (Valores Nulos): 13
Registros Finais: 183

=== COLUNAS DERIVADAS CRIADAS ===
Colunas atuais: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario', 'receita_total', 'mes', 'mes_nome', 'trimestre', 'ano', 'faixa_receita_item']

=== MÉTRICAS AGREGADAS ===
--- POR MÊS ---


,mes,receita_total,quantidade,n_vendas
0,1,120866.25,92,15
1,2,73895.74,70,12
2,3,123869.23,98,18
3,4,80636.64,48,7
4,5,132080.62,104,19



--- TOP PRODUTOS ---


,produto,receita_total
0,Notebook,374174.85
1,Tablet,335335.83
2,Smartphone,303255.94
3,Monitor,169554.67
4,Headset,48368.37



=== SEGMENTAÇÃO DE CLIENTES ===
--- Distribuição ---
segmento
Ouro      34
Prata     10
Bronze     6
Name: count, dtype: int64

--- Top 10 Clientes ---


,cliente,total_gasto,segmento
5,Cliente_006,65905.09,Ouro
35,Cliente_036,63433.04,Ouro
38,Cliente_039,59406.90,Ouro
19,Cliente_020,58459.54,Ouro
10,Cliente_011,56391.77,Ouro
33,Cliente_034,52035.60,Ouro
8,Cliente_009,50827.96,Ouro
23,Cliente_024,49146.94,Ouro
36,Cliente_037,46897.17,Ouro
18,Cliente_019,40731.05,Ouro



=== ESTATÍSTICAS NUMPY ===
media: 7051.07
desvio_padrao: 7809.92
maximo: 38833.20
minimo: 104.19
qtd_vendas_acima_media: 70.00

=== EXPORTAÇÃO DE ARQUIVOS ===
JSON gravado e lido com sucesso:
{
  "media": 7051.07,
  "desvio_padrao": 7809.92,
  "maximo": 38833.2,
  "minimo": 104.19,
  "qtd_vendas_acima_media": 70.0
}
Arquivos CSV salvos na pasta 'outputs/'.
